# 01 — Exploratory Data Analysis
LogisChain AI: supply chain network, temporal, and correlation analysis (Deliverable D2.1.2).

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import pandas as pd, numpy as np
from src.data.synthetic_generator import SupplyChainDataGenerator
gen = SupplyChainDataGenerator(seed=42)
nodes, edges = gen.generate_graph()
print(nodes.shape, edges.shape)
nodes.head()

(217, 34) (1223, 10)


,node_id,node_type,country,country_risk_score,natural_disaster_exposure,geopolitical_risk_score,current_ratio,debt_to_equity,ebitda_margin,interest_coverage,...,freight_cost_ratio,in_degree,out_degree,betweenness_centrality,clustering_coefficient,pagerank,pd_annual_true,default_12m,survival_duration_days,survival_event
0,SUPP_0000,supplier,China,72,0.20,0.30,0.980008,1.134138,0.012938,1.583040,...,0.131292,1,7,0.000000,0.214286,0.002060,0.123459,0,730.0,0
1,SUPP_0001,supplier,South Korea,82,0.10,0.28,1.682722,0.903321,0.258499,3.084793,...,0.051714,2,7,0.000000,0.111111,0.002100,0.014975,0,730.0,0
2,SUPP_0002,supplier,Mexico,58,0.15,0.20,1.264814,0.388181,0.219696,2.231905,...,0.097626,3,5,0.000000,0.142857,0.002148,0.004932,0,730.0,0
3,SUPP_0003,supplier,South Korea,82,0.10,0.28,1.728388,0.379941,0.107096,1.727641,...,0.101782,2,5,0.000154,0.142857,0.002115,0.005258,0,730.0,0
4,SUPP_0004,supplier,India,62,0.35,0.22,1.330483,1.201558,0.156065,4.611047,...,0.135406,2,8,0.000133,0.155556,0.002088,0.009239,0,730.0,0


## (a) Supply chain network visualisation
Node type counts, edge type counts, and network topology summary (degree/centrality distributions).

In [2]:
print(nodes.node_type.value_counts())
print()
print(edges.edge_type.value_counts())

node_type
supplier                 90
manufacturer             50
customer                 40
logistics_provider       20
port                     12
financial_institution     5
Name: count, dtype: int64

edge_type
transportation    634
material_flow     399
financial         183
ownership           7
Name: count, dtype: int64


In [3]:
import networkx as nx
G = nx.DiGraph()
G.add_nodes_from(nodes.node_id)
G.add_edges_from(edges[['src','dst']].itertuples(index=False, name=None))
print('Nodes:', G.number_of_nodes(), 'Edges:', G.number_of_edges())
print('Density:', nx.density(G))
degrees = pd.Series(dict(G.degree()))
print(degrees.describe())

Nodes: 217 Edges: 1223
Density: 0.026092336576207544
count    217.000000
mean      11.271889
std        8.849817
min        3.000000
25%        6.000000
50%        8.000000
75%       14.000000
max       48.000000
dtype: float64


## (b) Temporal pattern analysis
Seasonality decomposition of port throughput and freight rates.

In [4]:
ts = gen.generate_time_series(n_days=730)
ts.groupby('port')[['throughput_teu','freight_rate_usd_feu']].describe().T.head(20)

port                                Busan     Felixstowe        Hamburg  \
throughput_teu       count     730.000000     730.000000     730.000000   
                     mean    96551.984932  149262.673973   97424.324658   
                     std      8441.215038   20305.074257   11814.499285   
                     min     59100.000000   73834.000000   53434.000000   
                     25%     91201.000000  140853.750000   91240.500000   
                     50%     97043.500000  152106.000000   99111.500000   
                     75%    102519.500000  164051.000000  105909.000000   
                     max    114050.000000  181346.000000  117178.000000   
freight_rate_usd_feu count     730.000000     730.000000     730.000000   
                     mean     2220.572603    2301.072603    2282.715068   
                     std       434.457906     586.162046     550.047833   
                     min      1235.000000     929.000000    1129.000000   
                     25%      1907.250000    1910.000000    1925.500000   
                     50%      2204.500000    2226.000000    2215.500000   
                     75%      2509.750000    2583.500000    2557.750000   
                     max      4114.000000    4818.000000    4805.000000   

port                            Jebel Ali    Los Angeles         Manila  \
throughput_teu       count     730.000000     730.000000     730.000000   
                     mean   145009.560274  136161.239726  136687.646575   
                     std     15342.649607   13753.282909   15267.271460   
                     min     93234.000000   87407.000000   70464.000000   
                     25%    136509.000000  128895.000000  128655.000000   
                     50%    145746.000000  137762.000000  138051.000000   
                     75%    155959.750000  146360.000000  147409.250000   
                     max    175035.000000  162557.000000  163529.000000   
freight_rate_usd_feu count     730.000000     730.000000     730.000000   
                     mean     2277.021918    2280.197260    2273.290411   
                     std       543.864551     544.508514     550.064224   
                     min      1134.000000    1058.000000    1135.000000   
                     25%      1897.000000    1922.500000    1897.000000   
                     50%      2218.500000    2205.000000    2223.500000   
                     75%      2549.000000    2555.250000    2546.500000   
                     max      4746.000000    4510.000000    4738.000000   

port                           Manzanillo  Mumbai (JNPT)      Rotterdam  \
throughput_teu       count     730.000000     730.000000     730.000000   
                     mean   104214.475342  136793.049315  116223.891781   
                     std     11388.044723   12460.354641   11912.885796   
                     min     66066.000000  101216.000000   67708.000000   
                     25%     97734.250000  127113.750000  109601.750000   
                     50%    104867.000000  137055.500000  116893.000000   
                     75%    112485.250000  146998.000000  125143.500000   
                     max    128009.000000  164170.000000  137813.000000   
freight_rate_usd_feu count     730.000000     730.000000     730.000000   
                     mean     2293.916438    2229.365753    2251.501370   
                     std       538.446273     465.297066     433.062363   
                     min      1124.000000    1070.000000    1064.000000   
                     25%      1933.250000    1869.000000    1925.000000   
                     50%      2232.500000    2209.500000    2267.000000   
                     75%      2558.000000    2560.750000    2541.500000   
                     max      4821.000000    3710.000000    3499.000000   

port                               Santos       Shanghai      Singapore  
throughput_teu       count     730.000000     730.000000     730.000000  
                     mea

In [5]:
from src.features.temporal_features import build_temporal_features
port0 = ts[ts.port == ts.port.iloc[0]]
feats = build_temporal_features(port0, value_col='throughput_teu')
feats[['date','throughput_teu','roll_mean_30','ewma_30','month_sin','month_cos']].tail(10)

,date,throughput_teu,roll_mean_30,ewma_30,month_sin,month_cos
720,2024-12-21,74622.0,80082.533333,80056.273027,-2.449294e-16,1.0
721,2024-12-22,80626.0,80102.133333,80093.029606,-2.449294e-16,1.0
722,2024-12-23,83270.0,80157.766667,80297.995438,-2.449294e-16,1.0
723,2024-12-24,84474.0,80265.500000,80567.415087,-2.449294e-16,1.0
724,2024-12-25,81755.0,80202.000000,80644.033469,-2.449294e-16,1.0
725,2024-12-26,79419.0,80064.500000,80564.999051,-2.449294e-16,1.0
726,2024-12-27,79938.0,79933.566667,80524.547500,-2.449294e-16,1.0
727,2024-12-28,76163.0,79809.533333,80243.157338,-2.449294e-16,1.0
728,2024-12-29,77346.0,79793.333333,80056.243962,-2.449294e-16,1.0
729,2024-12-30,79194.0,79819.866667,80000.615319,-2.449294e-16,1.0


## (c) Correlation analysis: supply chain metrics vs. financial outcomes
Reproduces the logic behind Section A1.3's metric-financial-impact table.

In [6]:
from src.features.graph_features import build_entity_features, ENTITY_FEATURE_COLUMNS
feats_entity = build_entity_features(nodes)
corr_target = nodes.set_index('node_id')['pd_annual_true']
corrs = feats_entity.set_index('node_id')[ENTITY_FEATURE_COLUMNS].corrwith(corr_target)
corrs.sort_values(key=abs, ascending=False)

debt_to_equity                0.548106
ccc_days                      0.433738
inventory_turnover           -0.422414
supplier_concentration_hhi    0.205535
current_ratio                -0.173686
working_capital_ratio        -0.173686
geopolitical_risk_score       0.152251
otif_rate                    -0.142213
lead_time_std                 0.108061
ebitda_margin                -0.100110
out_degree                    0.064586
country_risk_score           -0.049515
lead_time_mean                0.044591
in_degree                    -0.036497
customer_concentration_hhi    0.026237
clustering_coefficient        0.024376
freight_cost_ratio            0.021600
natural_disaster_exposure     0.018226
interest_coverage             0.013520
pagerank                      0.010395
betweenness_centrality        0.000329
dtype: float64

## (d) Missing data assessment
The synthetic generator produces complete data by construction (no missingness); in a production deployment against real AIS/customs/ERP feeds, `src/data` would need an imputation strategy for the completeness gaps documented in Section A4.3 (e.g. AIS destination field only ~60-70% populated).

In [7]:
print('Missing values per column (nodes):')
print(nodes.isna().sum()[nodes.isna().sum() > 0])

Missing values per column (nodes):
Series([], dtype: int64)
